# Colab Training Notebook for TinyStoriesZh

This notebook is the conservative path: it does not change `train.py` or `prepare.py`, and just runs the current repo on Colab with a Google Drive-backed cache.

## 1. Mount Google Drive and configure paths

The cache directory is placed on Drive so dataset downloads and tokenizer artifacts can be reused across Colab sessions.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

REPO_URL = 'https://github.com/picasso250/autoresearch-win-rtx-zh.git'
BRANCH = 'autoresearch/apr4zh'
DRIVE_ROOT = Path('/content/drive/MyDrive')
WORKDIR = DRIVE_ROOT / 'colab' / 'autoresearch-win-rtx-zh'
REPO_DIR = WORKDIR / 'repo'
CACHE_DIR = WORKDIR / 'cache'

WORKDIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['AUTORESEARCH_CACHE_DIR'] = str(CACHE_DIR)

print(f'WORKDIR:   {WORKDIR}')
print(f'REPO_DIR:  {REPO_DIR}')
print(f'CACHE_DIR: {CACHE_DIR}')

## 2. Clone or refresh the repo

If the repo is already on Drive, this cell updates it to the selected branch.

In [ ]:
import shutil
import subprocess

def run(cmd, cwd=None):
    print('>', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

print('Repo ready:', REPO_DIR)

## 3. Install runtime dependencies

This keeps the Colab setup simple and reuses the PyTorch that is already available in Colab.

In [ ]:
import sys
import subprocess
import torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pyarrow>=21.0.0',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'tiktoken>=0.11.0',
], check=True)

print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

## 4. One-time data and tokenizer preparation

Run this once first. Because `AUTORESEARCH_CACHE_DIR` points to Drive, the parquet files and tokenizer stay cached for later sessions.

In [ ]:
env = os.environ.copy()
subprocess.run([sys.executable, 'prepare.py', '--dataset', 'tinystorieszh'], cwd=REPO_DIR, env=env, check=True)

## 5. Optional smoke test

This is the safest first run before spending time on the full 10-minute training budget.

In [ ]:
env = os.environ.copy()
subprocess.run([sys.executable, 'train.py', '--dataset', 'tinystorieszh', '--smoke-test'], cwd=REPO_DIR, env=env, check=True)

## 6. Full training run

Use this for a fresh 10-minute run from scratch.

In [ ]:
env = os.environ.copy()
subprocess.run([sys.executable, 'train.py', '--dataset', 'tinystorieszh'], cwd=REPO_DIR, env=env, check=True)

## 7. Resume from `checkpoint_pre_eval.pt`

Run this on later sessions after the checkpoint file exists in the repo directory.

In [ ]:
env = os.environ.copy()
subprocess.run([sys.executable, 'train.py', '--dataset', 'tinystorieszh', '--resume'], cwd=REPO_DIR, env=env, check=True)

## 8. Sample generation from the latest checkpoint

Useful for a quick sanity check after training.

In [ ]:
env = os.environ.copy()
subprocess.run([
    sys.executable, 'train.py', '--dataset', 'tinystorieszh', '--generate-only',
    '--prompt', '从前有一只小猫', '--max-new-tokens', '120'
], cwd=REPO_DIR, env=env, check=True)